# PDF reporting for HDAB id project

This script uses Python and the ReportLab toolkit https://www.reportlab.com/, the PyPDF module https://pypdf.readthedocs.io/en/stable/ to merge generated PDF files, and some basic modules. 

Dependencies:
- PyPDF v5.0 or higher (lower versions don't support functions to reduce PDF file size)
- PsycoPG
- Pandas
- ReportLab
- Pillow


To do:
- Reduce image file sizes
- Add argpase to run as a script
    - include wishlist option to supply a csv with list of morphospeciescodes to generate report for
    - include argument to set directory with images

## Import dependencies, fonts, and set working directories

In [39]:
import os
import time
import math
import sys
import linecache
import psycopg2
import pandas as pd
import reportlab
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.utils import ImageReader
from pypdf import PdfWriter
from PIL import Image
from datetime import date
from pathlib import Path

font_dir = "./fonts/" 
arial_reg = os.path.join(font_dir, "Arial.ttf")
arial_bold = os.path.join(font_dir, "Arial Bold.ttf")
arial_italic = os.path.join(font_dir, "Arial Italic.ttf")
arial_bolditalic = os.path.join(font_dir, "Arial Bold Italic.ttf")

pdfmetrics.registerFont(TTFont('Arial', arial_reg))
pdfmetrics.registerFont(TTFont('Arial-Bold', arial_bold))
pdfmetrics.registerFont(TTFont('Arial-Italic', arial_italic))
pdfmetrics.registerFont(TTFont('Arial-BoldItalic', arial_bolditalic))

img_dir = "/Volumes/Rubinoff_nas_412/DNA_voucher_photos/HDOA_Vouchers/"
#img_dir = "./test_imgs/"
database = 'aiven_hdab'

individualreports_dir = "Individual_reports"
os.makedirs(individualreports_dir, exist_ok=True) # creates output folder if it does not exist
combinedreports_dir = "Combined_reports"
os.makedirs(combinedreports_dir, exist_ok=True)

In [40]:
print(f"Using ReportLab version: {reportlab.Version}")

Using ReportLab version: 5.0.0


## PostgreSQL connection test function

In [35]:
def psqlconnectiontest(database):
    connectstringfile = str('.connectstring_' + database)
    if os.path.exists('./.connectstring_' + database) == False:
        sys.exit("Missing .connectstring file for database " + database + " : stopping")
    connectstring = linecache.getline(filename=connectstringfile, lineno=1).rstrip('\n')
    conn = psycopg2.connect(connectstring)
    if conn.closed == 0:
        print("Successfully connected to psql database")
    else:
        sys.exit("Could not connect to psql database: stopping") 
    conn = None

## Function to create a ReportLab PDFs with info from dataframe and add images

In [41]:
def pdfreporter(df, individualreports_dir, img_dir):
    for row in df.sort_values(by=['hdoareference', 'morphospeciescode']).itertuples():
        morphospeciescode = row.morphospeciescode
        print("Creating PDF report for: " + (morphospeciescode))
        # Create canvas with morphospeciescode filename in subfolder Reports
        filename = str(morphospeciescode) + ".pdf"
        full_path = os.path.join(individualreports_dir, filename)
        c = canvas.Canvas(full_path, pagesize=letter) # Letter size; width: 612, height: 792 points
        page_width, page_height = letter
        # Add header image
        UHIMlogo = "./logos/header1.png"
        img_width = 530
        img_height = 90
        x_coord = (page_width - img_width) / 2 # to center image on page
        c.drawImage(UHIMlogo, x_coord, 680, width=img_width, height=img_height, preserveAspectRatio=True)
        # Add title and date
        x_center = page_width / 2 # to center text on page
        c.setFont("Arial-Bold", 14) # font and size
        c.drawCentredString(x_center, 655, "University of Hawaiʻi Insect Museum")
        c.drawCentredString(x_center, 635, "Arthropod Identification Report")
        c.setFont("Arial", 10)
        today = date.today()
        c.drawCentredString(x_center, 620, ("Report generated: " + today.strftime("%B %d, %Y")))
        
        # Add information from dataframe
        c.setFont("Arial", 11) # keep 12 y value spacing between text with font size 11
        
        c.drawString(48, 586, "Project: Hawaiʻi Department of Agriculture and Biosecurity RFP-25-06-PI. Fiscal year 2025.")
    
        c.drawString(48, 567, "HDAB reference: " + str(row.hdoareference))
        c.drawString(48, 555, "Date collected: " + str(row.datecollected))
        c.drawString(48, 543, "Origin: " + str(row.origin))
        c.drawString(48, 531, "Intercepted host: " + str(row.host))
        c.drawString(48, 519, "Number of species in sample: " + str(row.speciescount))
    
        c.drawString(48, 495, "UHIM identification reference: " + str(morphospeciescode))
        
        if math.isnan(row.specimencount):
            specimencount = 'Not counted.'
        else: specimencount = round(row.specimencount)
        c.drawString(48, 483, "Number of specimens of this species: " + str(specimencount))
    
        c.drawString(48, 459, "Integrative identification:")
        if row.genusorlower == 1:
            c.setFont("Arial-Italic", 11)
        else: c.setFont("Arial", 11)
        c.drawString(169, 459, str(row.finalid))
        c.setFont("Arial", 11)
        c.drawString(48, 447, "Common name: " + str(row.finalidcommonname))
        c.drawString(48, 435, "Order: " + str(row.finalidorder))
        c.drawString(48, 423, "Family: " + str(row.finalidfamily))
    
        c.drawString(48, 399, "Identification notes: " + str(row.idnotes))
        c.drawString(48, 387, "Distribution notes: " + str(row.distributionnotes))
        
        # Add photo of morphotype
        # get a list of photos in the img_dir
        jpg_filenames = [
            f for f in os.listdir(img_dir) 
            if f.lower().endswith('.jpg') and os.path.isfile(os.path.join(img_dir, f))
        ]
        # Select image file names containing "morphospeciescode"
        morphospeciesimages = [item for item in jpg_filenames if morphospeciescode in item]
        if len(morphospeciesimages) == 0:
            print("   No images found, continuing without image")
        else:
            # 0. Select the first photo in the list to print
            speciesimage = os.path.join(img_dir, morphospeciesimages[0])
            # 1. Load and resize the image in RAM
            target_width = 500  # Set your desired maximum resolution width
            with Image.open(speciesimage) as img:
                w_percent = target_width / float(img.size[0]) # Calculate aspect ratio
                target_height = int((float(img.size[1]) * float(w_percent)))          
                # Resize using high-quality resampling
                resized_img = img.resize((target_width, target_height), Image.Resampling.LANCZOS)
                # 2. Save the resized image to an in-memory byte buffer
                img_buffer = io.BytesIO()
                # Explicitly set the format (e.g., JPEG or PNG) depending on your needs
                resized_img.save(img_buffer, format="JPEG", quality=85)
                img_buffer.seek(0)  # Reset buffer pointer to the beginning
            # 3. Print on page using reportlab (passing the buffer object instead of a file path)
            reportlab_image = ImageReader(img_buffer) # wrap the buffer in ImageReader
            img_width = 500
            img_height = 400
            x_coord = (page_width - img_width) / 2
            c.drawImage(reportlab_image, x_coord, 10, width=img_width, height=img_height, preserveAspectRatio=True)
            # 4. Add image filename underneath
            imgfilename = os.path.basename(speciesimage)
            c.setFont("Arial", 10) # font and size
            c.drawCentredString(x_center, 60, imgfilename)
            
        c.showPage() # finish page
        c.save() # construct and save file to .pdf
        print("   Successfully created report " + str(filename))

## Function to merge individual PDFs into one in separate folder

In [42]:
def combinepdf(individualreports_dir):
    today = date.today()
    pdf_with_folder = [
        os.path.join(individualreports_dir, f) 
        for f in os.listdir(individualreports_dir) 
        if f.lower().endswith('.pdf') and os.path.isfile(os.path.join(individualreports_dir, f))
    ]
    writer = PdfWriter()
    ordered_pdf_with_folder = sorted(pdf_with_folder)
    for pdf in ordered_pdf_with_folder:
        writer.append(pdf)

    writer.compress_identical_objects(remove_duplicates=True, remove_unreferenced=True)
    writer.write("./Combined_reports/" + today.strftime("%y%m%d") + "_combined_report.pdf")
    writer.close()
    print(f"Successfully merged {len(pdf_with_folder)} files")

## Run functions

In [43]:
start_time = time.perf_counter()

# test connection with database
try:
    psqlconnectiontest(database)
except Exception as e:
    print(f"An error occurred: {e}")
    
# pull data from postgres hdoa database and store in pandas dataframe 'df'
connectstringfile = str('.connectstring_' + database)
connectstring = linecache.getline(filename=connectstringfile, lineno=1).rstrip('\n')
conn = psycopg2.connect(connectstring)
sql = "SELECT * FROM public.pdfreport;"
df = pd.read_sql_query(sql, conn)
conn = None

# generate a PDF report for each morphospecies
try:
    pdfreporter(df, individualreports_dir, img_dir)
except Exception as e:
    print(f"An error occurred: {e}")

# combine reports into one PDF (keeps separate files too)
try:
    combinepdf(individualreports_dir)
except Exception as e:
    print(f"An error occurred: {e}")
    
end_time = time.perf_counter()
execution_time_seconds = end_time - start_time
minutes, seconds = divmod(execution_time_seconds, 60)
print(f"Execution time: {int(minutes)}m {seconds:.2f}s")

Successfully connected to psql database


/var/folders/zb/s73b85rs24j7t9n3wlzn05vc0000gn/T/ipykernel_70359/1040488143.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


Creating PDF report for: HDOA251015_009_M01
   Successfully created report HDOA251015_009_M01.pdf
Creating PDF report for: HDOA251015_009_M02
   Successfully created report HDOA251015_009_M02.pdf
Creating PDF report for: HDOA251015_009_M03
   Successfully created report HDOA251015_009_M03.pdf
Creating PDF report for: HDOA250917_034_M01
   Successfully created report HDOA250917_034_M01.pdf
Creating PDF report for: HDOA250917_034_M02
   No images found, continuing without image
   Successfully created report HDOA250917_034_M02.pdf
Creating PDF report for: HDOA250917_004_M01
   Successfully created report HDOA250917_004_M01.pdf
Creating PDF report for: HDOA250917_004_M02
   No images found, continuing without image
   Successfully created report HDOA250917_004_M02.pdf
Creating PDF report for: HDOA250917_006_M01
   Successfully created report HDOA250917_006_M01.pdf
Creating PDF report for: HDOA250917_035_M01
   Successfully created report HDOA250917_035_M01.pdf
Creating PDF report for: HDO

# STOP HERE

## Reduce resolution and files size in RAM:

In [32]:
import io
from pypdf import PdfReader, PdfWriter
from PIL import Image

def downsize_pdf_dimensions_in_memory(input_path, output_path, max_width=500, quality=80):
    writer = PdfWriter(clone_from=input_path)
    
    for page in writer.pages:
        for img in page.images:
            # 1. Open the raw PDF image bytes directly into Pillow
            pil_img = Image.open(io.BytesIO(img.data))
            
            # 2. Calculate the new dimensions maintaining the aspect ratio
            orig_width, orig_height = pil_img.size
            if orig_width > max_width:
                aspect_ratio = orig_height / orig_width
                new_width = max_width
                new_height = int(max_width * aspect_ratio)
                
                # 3. Resize the image using high-quality downscaling
                pil_img = pil_img.resize((new_width, new_height), Image.Resampling.LANCZOS)
            
            # 4. Save the resized image into an in-memory byte buffer
            img_buffer = io.BytesIO()
            
            # Convert to RGB mode if it's in RGBA (JPEGs don't support transparency)
            if pil_img.mode in ('RGBA', 'P'):
                pil_img = pil_img.convert('RGB')
                
            pil_img.save(img_buffer, format="JPEG", quality=quality)
            img_buffer.seek(0)
            
            # 5. Swap the old PDF image object with our new in-memory resized image
            img.replace(img_buffer, quality=quality)
            
    # Clean up and compress structural layout streams
    for page in writer.pages:
        page.compress_content_streams()
    writer.compress_identical_objects(remove_duplicates=True, remove_unreferenced=True)

    with open(output_path, "wb") as f:
        writer.write(f)
        
    print(f"Done! Saved layout-preserved PDF to {output_path}")

# Run the script
downsize_pdf_dimensions_in_memory("./Combined_reports/260925_combined_report.pdf", "./Combined_reports/optimized_resolution.pdf", max_width=800, quality=70)

TypeError: new_image shall be a PIL Image

## Using PyMuPDF to optimize combined PDF and reduce size

Probably don't need this.

In [ ]:
import pymupdf  # PyMuPDF

# Open the bulky PDF file
doc = pymupdf.open("./Combined_reports/260805_combined_report.pdf")

# Save with garbage collection and stream deflation
doc.save(
    "output.pdf",
    garbage=3,  # De-duplicates and drops unreferenced/duplicate objects
    deflate=True,  # Compresses uncompressed streams
    use_objstms=True,  # Packs metadata for extra space savings
)
doc.close()